1. Load processed data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("DatasetPreparation") \
    .master("local[*]") \
    .getOrCreate()

print("Spark OK:", spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 21:24:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 21:24:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark OK: <pyspark.sql.session.SparkSession object at 0x72422a1496a0>


In [2]:
df = spark.read.parquet("../data/processed/transactions")
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- deltaOrig: double (nullable = true)
 |-- deltaDest: double (nullable = true)
 |-- isBalanceErrorOrig: integer (nullable = true)
 |-- isBalanceErrorDest: integer (nullable = true)
 |-- amount_log: double (nullable = true)
 |-- type_index: double (nullable = true)



2. Feature selection

In [3]:
df = spark.read.parquet("../data/processed/transactions")

3. Define features

In [4]:
feature_cols = [
    "amount_log",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "deltaOrig",
    "deltaDest",
    "isBalanceErrorOrig",
    "isBalanceErrorDest",
    "type_index"
]

4. VectorAssembler

In [5]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = assembler.transform(df)

5. Final dataset (features + label)

In [6]:
df = df.select("features", col("isFraud").alias("label"))

6. TrainTest split

In [7]:
fraud_df = df.filter(col("label") == 1)
nonfraud_df = df.filter(col("label") == 0)

fraud_train, fraud_test = fraud_df.randomSplit([0.8, 0.2])
nonfraud_train, nonfraud_test = nonfraud_df.randomSplit([0.8, 0.2])

train_df = fraud_train.union(nonfraud_train)
test_df = fraud_test.union(nonfraud_test)

7. Sanity check

In [8]:
print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 5087910


[Stage 5:=====================================================>   (30 + 2) / 32]

Test: 1274710


8. Batch simulation 

In [9]:
for row in test_df.take(5):
    print(row)

[Stage 9:============================================>              (3 + 1) / 4]

Row(features=SparseVector(10, {0: 8.6246, 1: 5566.14, 4: 5566.14, 5: 5566.14, 6: 5566.14}), label=1)
Row(features=SparseVector(10, {0: 9.263, 1: 10539.37, 4: 10539.37, 5: 10539.37, 6: 10539.37}), label=1)
Row(features=SparseVector(10, {0: 9.824, 1: 18471.55, 4: 18471.55, 5: 18471.55, 6: 18471.55}), label=1)
Row(features=SparseVector(10, {0: 9.9216, 1: 20364.0, 4: 20364.0, 5: 20364.0, 6: 20364.0}), label=1)
Row(features=SparseVector(10, {0: 9.9793, 1: 21574.55, 4: 21574.55, 5: 21574.55, 6: 21574.55}), label=1)


9. Save train/test datasets

In [10]:
train_df.write.mode("overwrite").parquet("../data/processed/train")
test_df.write.mode("overwrite").parquet("../data/processed/test")